# Practice: PDP

In the PDP lesson we defined partial dependence as the expected prediction over the distribution of the other features, computed it by hand on five rows and went through its traps: a misleading axis, a flat curve under opposite effects, and plugging in values that never occur in the data. In this notebook the PDP is built by hand on real data, checked against `scikit-learn`, and every trap is reproduced with a number.

Every task has a cell with `...` that you fill in. The trainer tasks on this notebook are in the lesson “PDP: Practice” of the module “Explanation Methods for Machine Learning Models pt. 1” on the course platform.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence
from sklearn.inspection import PartialDependenceDisplay
from sklearn.neighbors import NearestNeighbors
%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## Data and model

The California Housing dataset ships with `sklearn`: 20,640 California block groups from the 1990 census, eight block-group features, and the target is the median house value in the block group in hundreds of thousands of dollars. The features: `MedInc` — median income (tens of thousands of dollars), `HouseAge` — median house age, `AveRooms` and `AveBedrms` — average number of rooms and bedrooms per house, `Population` — block-group population, `AveOccup` — average number of occupants per house, `Latitude` and `Longitude`.

We split the data into a training and a test part (80% and 20%, `random_state=42`) and train a gradient boosting model `HistGradientBoostingRegressor` with default parameters. For the methods of this notebook the model is a black box: all they need from it is the prediction `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"Test R^2 of the model: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

## 1. PDP by hand

$$\text{PD}(x_s) \approx \frac{1}{n}\sum_{i=1}^{n}\hat f\bigl(x_s, x_c^{(i)}\bigr).$$

For every grid value we put it into the feature's column in all $n$ rows, predict and average.

**Task 1.** Complete `pdp_by_hand` and check the result on 500 test rows against `partial_dependence(..., method="brute")`. Then compute the same curve with the default method for boosting, `method="recursion"`, and see how it differs.

In [ ]:
def pdp_by_hand(model, X, feature, grid):
    ...  # your code: len(grid) copies of X, in copy t the column feature equals grid[t]; predict and average over the rows of each copy
    return ...


X_pdp = X_test.sample(500, random_state=0)
brute = partial_dependence(model, X_pdp, ["MedInc"], kind="average", method="brute", grid_resolution=20)
grid = brute["grid_values"][0]
pd_hand = pdp_by_hand(model, X_pdp, "MedInc", grid)
print("Largest difference from brute:", np.abs(pd_hand - brute["average"][0]).max())

rec = partial_dependence(model, X_pdp, ["MedInc"], kind="average", method="recursion", grid_resolution=20)
shift = brute["average"][0] - rec["average"][0]
print("brute minus recursion: from", shift.min().round(3), "to", shift.max().round(3))
print("Mean of the target on the training data:", y_train.mean().round(3))

**Space for your conclusions:** does your curve match `brute`? By how much and why is `recursion` shifted against `brute`? Hint: `recursion` does not go over the rows of `X`, it walks down the trees and ignores the boosting model's initial prediction.

## 2. The cost of a PDP

The main cost of a PDP is the number of predictions: each of the $K$ grid values is plugged into all $n$ rows.

**Task 2.** Wrap the model in a counter of the rows it predicts and check that a PDP on a grid of $K = 20$ points over $n = 500$ rows costs $K \cdot n$ predictions.

In [ ]:
class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def predict(self, X):
        ...  # your code: add the number of rows of X to the counter
        return self.model.predict(X)


counted = CountingModel(model)
pdp_by_hand(counted, X_pdp, "MedInc", grid)
print("Predictions:", counted.rows, " K * n =", len(grid) * len(X_pdp))

## 3. PDP for several features and for a pair of features

**Task 3.** Draw `PartialDependenceDisplay` for the features `MedInc`, `AveOccup`, `HouseAge`, `AveRooms` and for the pair (`AveOccup`, `HouseAge`) on the same 500 rows, `method="brute"`. Which feature moves the average prediction most? Where on the two-dimensional map is the average prediction highest?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
display = ...  # your code: PartialDependenceDisplay.from_estimator for the four features and the pair
for a in display.axes_.ravel():
    if a is not None and a.get_ylabel() == "Partial dependence":
        a.set_ylabel("partial dependence")
fig.suptitle("PDP of the house value: California Housing, gradient boosting, 500 test rows")
fig.tight_layout()
plt.show()

## 4. The axis trap

A PDP shows the average prediction, and on an auto-scaled axis even a small change looks like a steep rise.

**Task 4.** Draw the PDP of `HouseAge` on two axes: auto-scaled and within the whole range of the model's predictions on the test data (from the minimum to the maximum). Compute what share of that range the span of the PDP takes.

In [ ]:
g_age = np.linspace(X_train["HouseAge"].quantile(0.05), X_train["HouseAge"].quantile(0.95), 20)
pd_age = pdp_by_hand(model, X_pdp, "HouseAge", g_age)
pred = model.predict(X_test)
share = ...  # your code: the span of the PDP divided by the span of the predictions
print(f"PDP span: {pd_age.max() - pd_age.min():.3f}, prediction span: {pred.max() - pred.min():.3f}, share: {share:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, lim in zip(axes, [None, (pred.min(), pred.max())]):
    ax.plot(g_age, pd_age, color=BLUE, lw=2)
    ax.set_xlabel("HouseAge")
    ax.set_ylabel("prediction, hundreds of thousands of dollars")
    if lim:
        ax.set_ylim(*lim)
axes[0].set_title("auto-scaled")
axes[1].set_title("the whole prediction range")
plt.show()

## 5. PDP-based Feature Importance

For a continuous feature the metric is the standard deviation of the PDP curve over the $k$ grid points:

$$I_{x_1} = \sqrt{\frac{1}{k-1}\sum_{i=1}^{k}\left(f(x_{1i}) - \frac{1}{k}\sum_{j=1}^{k}f(x_{1j})\right)^{2}}.$$

**Task 5.** Compute $I$ for all eight features on a grid of 20 points between the 5th and 95th percentiles of the feature and sort the features in decreasing order.

In [ ]:
def pdp_fi(pd_curve):
    return ...  # your code: the standard deviation of the curve with divisor k - 1


fi = {}
for feature in X.columns:
    g = np.linspace(X_train[feature].quantile(0.05), X_train[feature].quantile(0.95), 20)
    fi[feature] = pdp_fi(pdp_by_hand(model, X_pdp, feature, g))
pd.Series(fi, name="PDP-based FI").sort_values(ascending=False).round(3)

## 6. A flat PDP under a strong effect

We build synthetic data where for half of the objects the prediction grows with the feature and for the other half it falls: $y = 3\,s\,x_1 + \varepsilon$, where $x_1 \sim U(-1, 1)$, $s \in \{-1, +1\}$ is a group feature, $\varepsilon \sim N(0, 0.1^2)$. We train the same boosting model on $(x_1, s)$.

**Task 6.** Draw the PDP of $x_1$ and compute its PDP-based FI. Then draw the ICE curves of 60 objects colored by the group $s$, and compute the spread of an individual ICE curve (its standard deviation over the grid) averaged over the objects.

In [ ]:
rng = np.random.RandomState(0)
n = 2000
toy = pd.DataFrame({"x1": rng.uniform(-1, 1, n), "s": rng.choice([-1, 1], n)})
y_toy = 3 * toy["s"] * toy["x1"] + rng.normal(0, 0.1, n)
toy_model = HistGradientBoostingRegressor(random_state=0).fit(toy, y_toy)

g_toy = np.linspace(-0.95, 0.95, 20)
toy_ice = toy.iloc[:60]
X_rep = pd.concat([toy_ice] * len(g_toy), ignore_index=True)
X_rep["x1"] = np.repeat(g_toy, len(toy_ice))
ice_toy = toy_model.predict(X_rep).reshape(len(g_toy), len(toy_ice)).T
pd_toy = ...      # your code: the PDP of x1 on all rows of toy
ice_spread = ...  # your code: the mean spread of an individual ICE curve
print(f"PDP-based FI(x1) = {pdp_fi(pd_toy):.3f}, mean spread of an ICE curve = {ice_spread:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for s, color in [(-1, ORANGE), (1, BLUE)]:
    ax.plot(g_toy, ice_toy[toy_ice["s"].values == s].T, color=color, alpha=0.4, lw=1)
ax.plot(g_toy, pd_toy, color=INK, lw=3, label="PDP")
ax.set_xlabel("x1")
ax.set_ylabel("prediction")
ax.set_title("ICE of 60 objects (blue: s = +1, orange: s = -1) and the PDP")
ax.legend()
plt.show()

**Space for your conclusions:** what does the PDP-based FI say about the feature $x_1$, and what does ICE say?

## 7. A two-dimensional PDP and plugging in unrealistic values

Latitude and longitude in California Housing are related: block groups stretch along the coast, and most of the (latitude $\times$ longitude) rectangle is ocean and desert. A PDP plugs every pair of grid values into all rows without checking whether such a block group exists.

**Task 7.** Draw the two-dimensional PDP for the pair (`Longitude`, `Latitude`) on 300 test rows with a $20 \times 20$ grid and mark the real block groups on the map. Then compute the share of grid nodes with no block group nearby: the nearest block group of the whole dataset is farther than 0.25 degrees.

In [ ]:
X_geo = X_test.sample(300, random_state=0)
res = partial_dependence(model, X_geo, ["Longitude", "Latitude"], kind="average", method="brute", grid_resolution=20)
lon, lat = res["grid_values"]
surface = res["average"][0]

nodes = np.array([(a, b) for a in lon for b in lat])
nn = NearestNeighbors(n_neighbors=1).fit(X[["Longitude", "Latitude"]].values)
dist = nn.kneighbors(nodes)[0][:, 0]
empty_share = ...  # your code: the share of nodes whose nearest block group is farther than 0.25
print(f"Grid nodes with no block group within 0.25 degrees: {(dist > 0.25).sum()} of {len(nodes)} ({empty_share:.1%})")

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(lon, lat, surface.T, levels=12, cmap="Blues")
fig.colorbar(cs, ax=ax, label="partial dependence, hundreds of thousands of dollars")
ax.scatter(X["Longitude"], X["Latitude"], s=1, color=ORANGE, alpha=0.3, label="real block groups")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Two-dimensional PDP over longitude and latitude: California Housing, gradient boosting", fontsize=10)
ax.legend(loc="upper right", markerscale=8)
plt.show()

**Space for your conclusions:** what share of the PDP map is computed for block groups that do not exist? What does that mean for reading a two-dimensional PDP, and why does the next lesson of the module move on to ALE?